## Imports + configs

In [89]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score

from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMClassifier was fitted with feature names"
)


RANDOM_SEED = 42

DROP_CLASSES = True
DROP_WEAK_LINKS = {
    #"LeftHeelRises",
    "RightHeelRises",
    #"LeftKneeMovesInward",
    #"RightAsymmetricalWeightShift",
}
DROP_TIES = True

SCORES_PATH = "../../data/processed/20190108_scores_and_weak_links.csv"
FEATURES_PATH = "../../data/processed/AimoScore_WeakLink_big_scores.csv"


## Merge rare classes

In [90]:
# --- Rare class handling ---
MERGE_RARE = True
MIN_COUNT_PER_CLASS = 10   # anything below this becomes "<Region>__RARE"
RARE_SUFFIX = "__RARE"


def merge_rare_classes(y_sub: pd.Series, region_name: str, min_count: int, suffix: str = "__RARE"):
    """
    Merge labels that occur < min_count into a single label: f"{region_name}{suffix}"
    Returns:
      y_merged, rare_label, kept_labels, merged_labels
    """
    vc = y_sub.value_counts()
    rare_labels = vc[vc < min_count].index.tolist()
    rare_label = f"{region_name}{suffix}"

    if len(rare_labels) == 0:
        return y_sub.copy(), None, vc.index.tolist(), []

    y_merged = y_sub.copy()
    y_merged[y_merged.isin(rare_labels)] = rare_label

    kept_labels = [lbl for lbl in vc.index.tolist() if lbl not in rare_labels] + [rare_label]
    return y_merged, rare_label, kept_labels, rare_labels


## Load and build weakest link labels

In [91]:
scores = pd.read_csv(SCORES_PATH)
big = pd.read_csv(FEATURES_PATH)

NON_INDICATORS = {"ID", "Date", "SCORE"}
indicator_cols = [c for c in scores.columns if c not in NON_INDICATORS]

if DROP_CLASSES:
    indicator_cols = [c for c in indicator_cols if c not in DROP_WEAK_LINKS]

scores[indicator_cols] = scores[indicator_cols].apply(pd.to_numeric, errors="coerce")

row_max = scores[indicator_cols].max(axis=1)
ties = (scores[indicator_cols].eq(row_max, axis=0).sum(axis=1) > 1)

scores["WeakestLink"] = scores[indicator_cols].idxmax(axis=1)
scores["HasTie"] = ties

scores_labeled = scores.loc[~scores["HasTie"]].copy() if DROP_TIES else scores.copy()
scores_labeled = scores_labeled.dropna(subset=["WeakestLink"])

print("Unique WeakestLink classes:", scores_labeled["WeakestLink"].nunique())
display(scores_labeled["WeakestLink"].value_counts())


Unique WeakestLink classes: 13


WeakestLink
RightArmFallForward             452
LeftArmFallForward              322
RightKneeMovesOutward           275
RightShoulderElevation          245
ExcessiveForwardLean             94
ForwardHead                      92
LeftAsymmetricalWeightShift      71
LeftShoulderElevation            51
LeftKneeMovesOutward             47
RightKneeMovesInward             45
RightAsymmetricalWeightShift     19
LeftKneeMovesInward               3
LeftHeelRises                     2
Name: count, dtype: int64

## Merge features + build X/y (+ drop NaNs)

In [92]:
labels = scores_labeled[["ID", "WeakestLink"]]
data = big.merge(labels, on="ID", how="inner")

angle_cols = [c for c in data.columns if c.startswith("No_") and c.endswith("_Angle_Deviation")]
nasm_cols  = [c for c in data.columns if c.startswith("No_") and c.endswith("_NASM_Deviation")]
feature_cols = sorted(angle_cols) + sorted(nasm_cols)

X = data[feature_cols].apply(pd.to_numeric, errors="coerce")
y = data["WeakestLink"].astype(str)

mask = X.notna().all(axis=1)
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

print("X shape:", X.shape)
print("Min class count:", y.value_counts().min())


X shape: (1718, 38)
Min class count: 2


### Correct split

In [93]:
# After you create X, y, region (and apply the NaN mask)
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)
region = region.reset_index(drop=True)

ids = data.loc[mask, "ID"].reset_index(drop=True)

train_mask = ids.isin(train_ids).to_numpy()

X_train = X.loc[train_mask].reset_index(drop=True)
y_train = y.loc[train_mask].reset_index(drop=True)
region_train = region.loc[train_mask].reset_index(drop=True)

print("Train size:", X_train.shape, "Test size:", X.shape[0] - X_train.shape[0])


IndexError: Boolean index has wrong length: 1718 instead of 1724

## Define regions

In [59]:
UPPER = {
    "ForwardHead",
    "LeftArmFallForward",
    "RightArmFallForward",
    "LeftShoulderElevation",
    "RightShoulderElevation",
}

LOWER = {
    "LeftKneeMovesInward",
    "RightKneeMovesInward",
    "LeftKneeMovesOutward",
    "RightKneeMovesOutward",
    "LeftHeelRises",
    #"RightHeelRises",
}

OTHER = {
    "ExcessiveForwardLean",
    "LeftAsymmetricalWeightShift",
    "RightAsymmetricalWeightShift",
}

def to_region(lbl: str) -> str:
    if lbl in UPPER: return "Upper"
    if lbl in LOWER: return "Lower"
    if lbl in OTHER: return "Other"
    return "Other"   # safe fallback

region = y.map(to_region)

print(region.value_counts())


WeakestLink
Upper    1181
Lower     373
Other     170
Name: count, dtype: int64


## Define a helper to train + save one expert

In [60]:
def make_expert():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=600,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ])

def train_save_expert(name, region_name, X_sub, y_sub, feature_cols, out_path,
                      merge_rare=True, min_count=10, rare_suffix="__RARE"):
    print(f"\n=== Training {name} ===")
    print("Original classes:", sorted(y_sub.unique()))
    print("Original counts:\n", y_sub.value_counts())

    # --- Merge rare classes (optional) ---
    if merge_rare:
        y_sub_merged, rare_label, kept_labels, merged_labels = merge_rare_classes(
            y_sub=y_sub, region_name=region_name, min_count=min_count, suffix=rare_suffix
        )
        if merged_labels:
            print(f"\nMerged rare classes (<{min_count}) into '{rare_label}':")
            print("Merged labels:", merged_labels)
            print("\nNew counts:\n", y_sub_merged.value_counts())
        else:
            print(f"\nNo rare classes to merge (<{min_count}).")

        y_train = y_sub_merged
    else:
        y_train = y_sub

    # --- Choose folds safely ---
    min_count_train = int(y_train.value_counts().min())
    n_splits = min(5, min_count_train)

    if n_splits < 2:
        print(f"Not enough samples for CV (min_count={min_count_train}). Training final model only.")
        final_model = make_expert()
        final_model.fit(X_sub, y_train)

        joblib.dump(
            {"model": final_model, "feature_cols": feature_cols, "classes": sorted(pd.unique(y_train))},
            out_path
        )
        print(f"Saved: {out_path}")
        return

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    f1s = []

    for fold, (tr, te) in enumerate(skf.split(X_sub, y_train), start=1):
        Xtr, Xte = X_sub.iloc[tr], X_sub.iloc[te]
        ytr, yte = y_train.iloc[tr], y_train.iloc[te]

        m = make_expert()
        m.fit(Xtr, ytr)
        pred = m.predict(Xte)

        f1 = f1_score(yte, pred, average="macro")
        f1s.append(f1)
        print(f"fold {fold} macro_f1 = {f1:.4f}")

    print(f"{name} mean macro_f1 = {np.mean(f1s):.4f}")

    # --- Fit final + save ---
    final_model = make_expert()
    final_model.fit(X_sub, y_train)

    merge_info = {
        "merged_labels": merged_labels if merge_rare else [],
        "rare_label": rare_label if merge_rare else None,
        "min_count": min_count,
    }

    joblib.dump(
        {
            "model": final_model,
            "feature_cols": feature_cols,
            "classes": sorted(pd.unique(y_train)),
            "merge_rare": merge_rare,
            "min_count_per_class": min_count,
            "rare_suffix": rare_suffix,
            "region_name": region_name,
            "merge_info": merge_info,   # <-- add this
        },
        out_path
    )

    print(f"Saved: {out_path}")



## Train experts per region + save

In [61]:
X_upper = X_train.loc[region_train == "Upper"].reset_index(drop=True)
y_upper = y_train.loc[region_train == "Upper"].reset_index(drop=True)

X_lower = X_train.loc[region_train == "Lower"].reset_index(drop=True)
y_lower = y_train.loc[region_train == "Lower"].reset_index(drop=True)

X_other = X_train.loc[region_train == "Other"].reset_index(drop=True)
y_other = y_train.loc[region_train == "Other"].reset_index(drop=True)


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).

In [53]:
train_save_expert(
    name="Upper expert",
    region_name="Upper",
    X_sub=X_upper, y_sub=y_upper,
    feature_cols=feature_cols,
    out_path="upper_expert.pkl",
    merge_rare=MERGE_RARE,
    min_count=MIN_COUNT_PER_CLASS,
    rare_suffix=RARE_SUFFIX
)

train_save_expert(
    name="Lower expert",
    region_name="Lower",
    X_sub=X_lower, y_sub=y_lower,
    feature_cols=feature_cols,
    out_path="lower_expert.pkl",
    merge_rare=MERGE_RARE,
    min_count=MIN_COUNT_PER_CLASS,
    rare_suffix=RARE_SUFFIX
)

train_save_expert(
    name="Other expert",
    region_name="Other",
    X_sub=X_other, y_sub=y_other,
    feature_cols=feature_cols,
    out_path="other_expert.pkl",
    merge_rare=MERGE_RARE,
    min_count=MIN_COUNT_PER_CLASS,
    rare_suffix=RARE_SUFFIX
)



=== Training Upper expert ===
Original classes: ['ForwardHead', 'LeftArmFallForward', 'LeftShoulderElevation', 'RightArmFallForward', 'RightShoulderElevation']
Original counts:
 WeakestLink
RightArmFallForward       364
LeftArmFallForward        256
RightShoulderElevation    198
ForwardHead                73
LeftShoulderElevation      39
Name: count, dtype: int64

No rare classes to merge (<10).
fold 1 macro_f1 = 0.6355
fold 2 macro_f1 = 0.5144
fold 3 macro_f1 = 0.5896
fold 4 macro_f1 = 0.5995
fold 5 macro_f1 = 0.6765
Upper expert mean macro_f1 = 0.6031
Saved: upper_expert.pkl

=== Training Lower expert ===
Original classes: ['LeftHeelRises', 'LeftKneeMovesInward', 'LeftKneeMovesOutward', 'RightKneeMovesInward', 'RightKneeMovesOutward']
Original counts:
 WeakestLink
RightKneeMovesOutward    220
LeftKneeMovesOutward      37
RightKneeMovesInward      36
LeftKneeMovesInward        3
LeftHeelRises              1
Name: count, dtype: int64

Merged rare classes (<10) into 'Lower__RARE':
Merg